# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shenone77/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

Content refresh triage for editors with a fixed review budget.

## Setup

Open this notebook from the repository or its Colab badge. The analysis uses FlyRank's bundled anonymized sample.

In [1]:
from pathlib import Path
import subprocess
import sys

if "google.colab" in sys.modules:
    root = Path("/content/flyrank-ml-internship")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Shenone77/flyrank-ml-internship.git", str(root)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2", "scikit-learn>=1.4"], check=True)
else:
    start = Path.cwd().resolve()
    root = next(
        (path for path in (start, *start.parents) if (path / "work/notebooks/refresh_triage.py").is_file()),
        None,
    )
    if root is None:
        raise FileNotFoundError("Open the notebook inside a clone of the assignment repository.")

sys.path.insert(0, str(root / "work/notebooks"))
from refresh_triage import load_pages

pages = load_pages(root)
print(f"Loaded {len(pages):,} eligible pages from the FlyRank starter sample.")

Loaded 13,412 eligible pages from the FlyRank starter sample.


## 1. My lane and why

I picked Lane 2, **Refresh / Content Opportunity Scoring**. Week 1 showed how hard it is to fill a review queue from a rigid stale-and-visible rule. A practical alternative is to ask which previously visible pages are at risk of a large *absolute* loss as well as a large percentage loss. That focuses the editor's time on declines worth investigating.

## 2. The decision, action, and cost

**Research question.** Among pages with at least 300 search impressions in the preceding 30 days, which pages should an editor review first for a drop of at least 30% and at least 150 impressions in the following 30 days?

One row represents one pseudonymized page from one client at the decision cutoff. The output is a ranked list for the first 50 manual checks. The editor looks at indexing, seasonality, query mix, and recent edits before proposing a refresh. A false alarm takes a review slot away from another page. A missed decline delays the investigation of a page that lost meaningful visibility. Past exposure, click-through rate, and page age may help order the queue because their effects can vary together.

## 3. Quick look at the data

The public starter slice has two adjacent 30-day impression windows. The preceding window defines eligibility. The following window provides the observed outcome. These three descriptive numbers motivate a ranked queue.

In [2]:
print(f"Eligible pages: {len(pages):,}")
print(f"Eligible pages with a material drop: {pages['material_drop'].sum():,}")
print(f"Observed material-drop share: {pages['material_drop'].mean():.1%}")
assert pages["material_drop"].nunique() == 2

Eligible pages: 13,412
Eligible pages with a material drop: 6,643
Observed material-drop share: 49.5%


## 4. Careful interpretation

The two thresholds are editorial decision rules applied to observed impression counts. They distinguish a substantial loss from a volatile small denominator. The share above describes this selected snapshot. A grouped holdout can test transfer between clients within this snapshot, and a later snapshot would be needed for a time-based generalization claim. A successful queue would support triage. Estimating the effect of a refresh would require an intervention design.

## Self-check

The code measures the three figures from the bundled sample. The proposal identifies a user, a given review budget, a measurable outcome, and the cost of a wrong pick. The displayed figures are aggregate.